# 🛒 Agente de Atendimento com IA — Demonstração
Este notebook **não contém a lógica**: ele baixa o código do GitHub e usa o pacote `agente_olist`.
Assim, o que você melhora no repositório aparece aqui automaticamente.

**Antes de começar (uma vez só):**
1. Crie uma chave grátis em https://aistudio.google.com/apikey
2. Aqui no Colab, clique no ícone 🔑 (*Secrets*) na barra lateral → **Add new secret**
   - Nome: `GEMINI_API_KEY` · Valor: sua chave · ative *Notebook access*

## 1. Baixar o código do GitHub e instalar

In [ ]:
REPO = "https://github.com/IsadoraRibeiro20/agente-atendimento-olist.git"
!git clone -q $REPO /content/projeto 2>/dev/null || git -C /content/projeto pull -q
%cd /content/projeto
!pip install -q -e .

import sys; sys.path.insert(0, "/content/projeto/src")

## 2. Baixar os dados da Olist
Se pedir login: crie conta grátis no Kaggle → *Settings* → *API* → *Create New Token*, e adicione
`KAGGLE_USERNAME` e `KAGGLE_KEY` nos Secrets. Ou baixe o .zip em
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce e envie os CSVs para `data/raw/`.

In [ ]:
import os
from google.colab import userdata
for k in ["KAGGLE_USERNAME", "KAGGLE_KEY"]:
    try: os.environ[k] = userdata.get(k)
    except Exception: pass

!python scripts/baixar_dados.py

## 3. Testar as ferramentas SEM IA (não gasta cota)
Cada ferramenta é uma função Python comum. Entenda o que elas devolvem antes de ligar a IA.

In [ ]:
from agente_olist import ferramentas
from agente_olist.dados import carregar_base
from agente_olist.preditivo import ModeloAtraso

base = carregar_base()
modelo = ModeloAtraso(base)
ferramentas.configurar(base, modelo_atraso=modelo)
print(f"Pedidos: {len(base.pedidos):,} | AUC do modelo de atraso: {modelo.auc:.3f}")
base.pedidos.head()

In [ ]:
exemplo = base.pedidos.order_id.iloc[0]
ferramentas.consultar_pedido(exemplo)

In [ ]:
ferramentas.prever_atraso(exemplo)

In [ ]:
ferramentas.conciliar_pagamentos()

In [ ]:
ferramentas.resumo_vendas("2018-03")

## 4. Ligar a IA: o agente
Repare nas linhas 🔧 — são as ferramentas que **a própria IA decidiu chamar**.

In [ ]:
from agente_olist.agente import Agente

agente = Agente()
ferramentas.configurar(base, modelo_atraso=modelo, cliente_ia=agente.cliente)

print(agente.perguntar(f"Oi! Qual o status do meu pedido {exemplo[:8]}? Tem risco de atraso?"))

In [ ]:
# O agente lembra da conversa (histórico)
print(agente.perguntar("E quanto eu paguei nele?"))

In [ ]:
print(agente.perguntar("Quais os principais motivos de reclamação dos clientes?"))

In [ ]:
print(agente.perguntar("Faça a conciliação dos pagamentos, gere um relatório executivo com vendas, "
                       "conciliação e reclamações, e salve."))
!ls relatorios/

## 5. Chat de atendimento (página web)
Gera um link público `https://....gradio.live` — abra no celular e converse como se fosse cliente.
O link funciona enquanto este notebook estiver ativo.

In [ ]:
import gradio as gr
from agente_olist.app import EXEMPLOS

agente.limpar()
def responder(mensagem, historico):
    if not historico: agente.limpar()
    return agente.perguntar(mensagem)

gr.ChatInterface(responder, title="🛒 Atendimento Inteligente — Olist", examples=EXEMPLOS).launch(share=True)

---
💡 **Limite grátis:** se aparecer erro `429 RESOURCE_EXHAUSTED`, espere ~1 minuto (limite por minuto)
ou volte amanhã (limite diário).